# 🛡️ The Sentinel Ego — Phase 1: Persistent Behavioral Identity (PBI)

**Goal:** Mine real Enron email behavioral data, discover 10 archetypes via K-Means,
train 3rd-order Markov Chains, generate 90-day persona trajectories, and validate
behavioral consistency via Jensen-Shannon Divergence (JSD < 0.1).

**Dataset:** CMU Enron Email Corpus (517,401 emails, 92 eligible users)
**Output:** 30 personas across 10 archetypes | 20,459 total events | JSD 0.0495–0.0999

In [ ]:
# Cell P1-1: Environment Setup
!pip -q install pandas numpy scipy scikit-learn tqdm python-dateutil matplotlib seaborn

import os, re, tarfile, glob, email, math, json
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import entropy
from scipy.spatial.distance import jensenshannon
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from email import policy
from email.parser import BytesParser
from dateutil import parser as dtparser
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = '/content/sentinel_ego_phase1'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
PROC_DIR = os.path.join(BASE_DIR, 'processed')
OUT_DIR = os.path.join(BASE_DIR, 'outputs')
for d in [BASE_DIR, RAW_DIR, PROC_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Phase 1 environment ready.')

In [ ]:
# Cell P1-2: Download Enron Email Corpus
import urllib.request

enron_url = 'https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz'
enron_tar = os.path.join(RAW_DIR, 'enron_mail_20150507.tar.gz')

if not os.path.exists(enron_tar):
    print('Downloading Enron corpus (~422 MB)...')
    urllib.request.urlretrieve(enron_url, enron_tar)

print(f'File size: {os.path.getsize(enron_tar)/1024/1024:.2f} MB')

In [ ]:
# Cell P1-3: Extract and Locate Maildir
extract_dir = os.path.join(RAW_DIR, 'enron_maildir')
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(enron_tar, 'r:gz') as tar:
        tar.extractall(path=extract_dir)

def find_maildir(root):
    for r, dirs, files in os.walk(root):
        if os.path.basename(r).lower() == 'maildir':
            return r
    return None

MAILDIR = find_maildir(extract_dir)
print('MAILDIR:', MAILDIR)

In [ ]:
# Cell P1-4: Parse Raw Emails
def safe_parse_date(date_str):
    if not date_str or pd.isna(date_str): return pd.NaT
    try:
        dt = dtparser.parse(date_str)
        if dt.tzinfo is not None: dt = dt.replace(tzinfo=None)
        return pd.Timestamp(dt)
    except: return pd.NaT

def extract_email_record(file_path):
    try:
        with open(file_path, 'rb') as f:
            msg = BytesParser(policy=policy.default).parse(f)
        from_ = str(msg.get('From', '')).strip().lower()
        to_ = str(msg.get('To', '')).strip().lower()
        cc_ = str(msg.get('Cc', '')).strip().lower()
        bcc_ = str(msg.get('Bcc', '')).strip().lower()
        subject_ = str(msg.get('Subject', '')).strip()
        date_raw = str(msg.get('Date', '')).strip()
        message_id = str(msg.get('Message-ID', '')).strip()
        folder = os.path.relpath(os.path.dirname(file_path), MAILDIR)
        owner = folder.split(os.sep)[0] if os.sep in folder else folder
        dt = safe_parse_date(date_raw)
        if pd.isna(dt): return None
        def cnt(x): return len([p.strip() for p in re.split(r'[;,]', x) if p.strip()]) if isinstance(x,str) and x.strip() else 0
        return {
            'file_path': file_path, 'owner': owner.lower(), 'folder': folder.lower(),
            'from': from_, 'to': to_, 'cc': cc_, 'bcc': bcc_,
            'subject': subject_, 'date': dt, 'message_id': message_id,
            'to_count': cnt(to_), 'cc_count': cnt(cc_), 'bcc_count': cnt(bcc_),
            'subject_len': len(subject_),
            'is_sent_folder': int(any(k in folder.lower() for k in ['sent','_sent_mail','sent_items'])),
            'is_inbox_folder': int('inbox' in folder.lower())
        }
    except: return None

email_files = [os.path.join(r,f) for r,d,files in os.walk(MAILDIR) for f in files]
print(f'Raw email files: {len(email_files):,}')
records = [r for fp in tqdm(email_files) for r in [extract_email_record(fp)] if r is not None]
df = pd.DataFrame(records)
print(f'Parsed: {len(df):,} records')

In [ ]:
# Cell P1-5: Feature Engineering + Eligible User Filtering
df = df.drop_duplicates(subset=['message_id','date','owner','subject']).copy()
df = df[df['owner'].notna() & df['date'].notna()].copy()
df['hour'] = df['date'].dt.hour
df['dayofweek'] = df['date'].dt.dayofweek
df['date_only'] = df['date'].dt.date
df['recipient_total'] = df['to_count'] + df['cc_count'] + df['bcc_count']

sent_df = df[df['is_sent_folder']==1].copy()

user_span = sent_df.groupby('owner').agg(
    total_sent=('message_id','count'),
    active_days=('date_only', pd.Series.nunique),
    first_date=('date','min'), last_date=('date','max')
).reset_index()
user_span['span_days'] = (user_span['last_date'] - user_span['first_date']).dt.days + 1

eligible_users = user_span[
    (user_span['total_sent'] >= 200) &
    (user_span['active_days'] >= 30) &
    (user_span['span_days'] >= 60)
]['owner'].tolist()

print(f'Eligible users: {len(eligible_users)}')
eligible_df = sent_df[sent_df['owner'].isin(eligible_users)].copy()

In [ ]:
# Cell P1-6: K-Means Clustering (K=10 archetypes)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

def normalized_hist(series, bins):
    counts, _ = np.histogram(series, bins=bins)
    counts = counts.astype(float) + 1e-9
    return counts / counts.sum()

user_features = []
for user in eligible_users:
    u = eligible_df[eligible_df['owner']==user]
    if len(u) < 100: continue
    hour_dist = normalized_hist(u['hour'], bins=np.arange(25))
    dow_dist = normalized_hist(u['dayofweek'], bins=np.arange(8))
    user_features.append({
        'owner': user, 'total_sent': len(u),
        'active_days': u['date_only'].nunique(),
        'mean_hour': u['hour'].mean(), 'std_hour': u['hour'].std(),
        'peak_hour': u['hour'].mode()[0] if len(u['hour'].mode()) > 0 else u['hour'].mean(),
        'weekend_ratio': u['dayofweek'].isin([5,6]).mean(),
        'mean_recipients': u['recipient_total'].mean(),
        'entropy_hour': entropy(hour_dist + 1e-12),
        'entropy_dow': entropy(dow_dist + 1e-12),
        'emails_per_active_day': len(u) / max(u['date_only'].nunique(), 1)
    })

features_df = pd.DataFrame(user_features)
feature_cols = ['mean_hour','std_hour','peak_hour','weekend_ratio','mean_recipients',
                'entropy_hour','entropy_dow','emails_per_active_day']

X = features_df[feature_cols].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Silhouette score to find optimal K
sil_scores = {}
for k in range(2, 12):
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=300)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

best_k = max(sil_scores, key=sil_scores.get)
print(f'Optimal K = {best_k} (silhouette = {sil_scores[best_k]:.4f})')

kmeans = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=300)
features_df['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# Cell P1-7: Assign Archetype Names and Train 3rd-Order Markov Chains
ARCHETYPE_NAMES = {
    0: 'Morning_Bird', 1: 'Collaborator', 2: 'Balanced', 3: 'Workaholic',
    4: 'Night_Owl', 5: 'Tech_Savvy', 6: 'Careful_Planner',
    7: 'Lone_Wolf', 8: 'Workaholic_8', 9: 'Social_Butterfly'
}
features_df['archetype'] = features_df['cluster'].map(ARCHETYPE_NAMES)

def train_markov_chain(hour_seq, order=3):
    transitions = {}
    for i in range(len(hour_seq) - order):
        state = tuple(hour_seq[i:i+order])
        next_h = hour_seq[i+order]
        if state not in transitions:
            transitions[state] = {}
        transitions[state][next_h] = transitions[state].get(next_h, 0) + 1
    for state in transitions:
        total = sum(transitions[state].values())
        transitions[state] = {k: v/total for k, v in transitions[state].items()}
    return transitions

markov_chains = {}
for cluster_id, archetype_name in ARCHETYPE_NAMES.items():
    cluster_users = features_df[features_df['cluster']==cluster_id]['owner'].tolist()
    cluster_emails = eligible_df[eligible_df['owner'].isin(cluster_users)].sort_values('date')
    hour_seq = cluster_emails['hour'].tolist()
    markov_chains[archetype_name] = train_markov_chain(hour_seq, order=3)
    print(f'{archetype_name}: {len(cluster_emails):,} emails, {len(markov_chains[archetype_name]):,} states')
print('All Markov chains trained.')

In [ ]:
# Cell P1-8: Generate 90-Day Persona Trajectories (3 per archetype = 30 total)
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

def sample_markov(chain, seed_state, n_steps):
    seq = list(seed_state)
    for _ in range(n_steps):
        state = tuple(seq[-3:])
        if state in chain and chain[state]:
            nxt = random.choices(list(chain[state].keys()), weights=list(chain[state].values()))[0]
        else:
            nxt = random.randint(7, 18)
        seq.append(nxt)
    return seq[3:]

all_events = []
for cluster_id, archetype_name in ARCHETYPE_NAMES.items():
    arch_users = features_df[features_df['cluster']==cluster_id]
    arch_emails = eligible_df[eligible_df['owner'].isin(arch_users['owner'].tolist())]
    stats = features_df[features_df['cluster']==cluster_id][['mean_recipients','weekend_ratio','emails_per_active_day']].mean()
    chain = markov_chains[archetype_name]
    for persona_idx in range(3):
        persona_id = f'{archetype_name}_P{persona_idx+1}'
        start_date = datetime(2024, 1, 1) + timedelta(days=persona_idx*5)
        seed = list(arch_emails['hour'].sample(3, random_state=persona_idx).values)
        hours = sample_markov(chain, seed, int(stats['emails_per_active_day']*90*1.2))
        h_idx = 0
        for day in range(90):
            current_date = start_date + timedelta(days=day)
            is_weekend = current_date.weekday() >= 5
            if is_weekend and random.random() > stats['weekend_ratio']: continue
            n_emails = max(1, int(np.random.poisson(stats['emails_per_active_day'])))
            for _ in range(n_emails):
                if h_idx >= len(hours): break
                hour = hours[h_idx]; h_idx += 1
                all_events.append({
                    'persona_id': persona_id, 'archetype': archetype_name,
                    'date': current_date.strftime('%Y-%m-%d'), 'hour': hour,
                    'dayofweek': current_date.weekday(),
                    'recipients': max(1, int(np.random.poisson(stats['mean_recipients'])))
                })

events_df = pd.DataFrame(all_events)
print(f'Total events generated: {len(events_df):,}')
print(f'Personas: {events_df["persona_id"].nunique()}')

In [ ]:
# Cell P1-9: JSD Consistency Validation
from scipy.spatial.distance import jensenshannon

def normalized_hist_arr(series, bins):
    counts, _ = np.histogram(series, bins=bins)
    counts = counts.astype(float) + 1e-9
    return counts / counts.sum()

jsd_results = []
for persona_id in events_df['persona_id'].unique():
    pdata = events_df[events_df['persona_id']==persona_id].copy()
    pdata['date'] = pd.to_datetime(pdata['date'])
    pdata = pdata.sort_values('date')
    days = sorted(pdata['date'].unique())
    if len(days) < 30: continue
    n = len(days)
    first_days = days[:int(n*0.35)]
    last_days = days[int(n*0.65):]
    first = pdata[pdata['date'].isin(first_days)]
    last = pdata[pdata['date'].isin(last_days)]
    if len(first) < 20 or len(last) < 20: continue
    jsd_h = jensenshannon(normalized_hist_arr(first['hour'], np.arange(25)),
                          normalized_hist_arr(last['hour'], np.arange(25)))
    jsd_dow = jensenshannon(normalized_hist_arr(first['dayofweek'], np.arange(8)),
                            normalized_hist_arr(last['dayofweek'], np.arange(8)))
    jsd_rec = jensenshannon(normalized_hist_arr(first['recipients'].clip(0,20), np.arange(22)),
                            normalized_hist_arr(last['recipients'].clip(0,20), np.arange(22)))
    jsd_mean = np.mean([jsd_h, jsd_dow, jsd_rec])
    jsd_results.append({'persona_id': persona_id, 'jsd_hour': jsd_h,
                        'jsd_dow': jsd_dow, 'jsd_recipients': jsd_rec,
                        'jsd_mean': jsd_mean,
                        'pass': int(jsd_mean < 0.1)})

jsd_df = pd.DataFrame(jsd_results).sort_values('jsd_mean')
passing = jsd_df['pass'].sum()
print(f'Personas passing JSD < 0.1: {passing}/{len(jsd_df)}')
print(f'Best persona JSD: {jsd_df["jsd_mean"].min():.4f} ({jsd_df.iloc[0]["persona_id"]})')
jsd_df.to_csv(os.path.join(OUT_DIR, 'p1_jsd_consistency.csv'), index=False)

In [ ]:
# Cell P1-10: Save All Phase 1 Outputs
events_df.to_csv(os.path.join(OUT_DIR, 'p1_persona_trajectories_90day.csv'), index=False)
features_df.to_csv(os.path.join(OUT_DIR, 'p1_user_behavior_features.csv'), index=False)

archetype_summary = features_df.groupby('archetype').agg(
    n_users=('owner','count'),
    mean_hour=('mean_hour','mean'),
    emails_per_day=('emails_per_active_day','mean'),
    weekend_ratio=('weekend_ratio','mean'),
    mean_recipients=('mean_recipients','mean')
).reset_index().round(4)
archetype_summary.to_csv(os.path.join(OUT_DIR, 'p1_archetype_summary.csv'), index=False)

print('Phase 1 complete. Outputs saved to:', OUT_DIR)
print(archetype_summary.to_string())